In [4]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # needed to enable 3D projection
from matplotlib.animation import FuncAnimation


# 1. GRADIENT DESCENT
def gradient_descent(x, y, lr=0.001, number_iteration=1000):
    m = x.shape[0]             
    w = 0.0                     
    b = 0.0                    
    cost_history = []           
    w_history = []               
    b_history = []               

    for i in range(number_iteration):
        y_pred = w * x + b                             
        cost = (1 / m) * np.sum((y_pred - y) ** 2)     
        cost_history.append(cost)
        w_history.append(w)
        b_history.append(b)

        dw = (2 / m) * np.sum((y_pred - y) * x)          
        db = (2 / m) * np.sum(y_pred - y)                

        w -= lr * dw
        b -= lr * db

    return w, b, cost_history, w_history, b_history


x = np.array([1, 2, 3, 4, 5])
y = np.array([10, 40, 50, 78, 83])

w, b, cost_history, w_history, b_history = gradient_descent(x, y, lr=0.01, number_iteration=1000)
print(f"Final weight: {w:.4f}, Final bias: {b:.4f}")

m = x.shape[0]

frame_step = 40
frames = list(range(0, len(w_history), frame_step))

# 2. PRECOMPUTE EVERYTHING THE PANELS NEED
w_range = np.linspace(min(w_history) - 5, max(w_history) + 5, 200)
cost_for_w = [(1 / m) * np.sum((w_val * x + b - y) ** 2) for w_val in w_range]

# Panel 3: Cost vs b (w held at its final value)
b_range = np.linspace(min(b_history) - 5, max(b_history) + 5, 200)
cost_for_b = [(1 / m) * np.sum((w * x + b_val - y) ** 2) for b_val in b_range]

# Panels 4 & 5: full (w, b) cost surface, used by both the 3D plot and the contour plot
w_grid = np.linspace(min(w_history) - 5, max(w_history) + 5, 60)
b_grid = np.linspace(min(b_history) - 5, max(b_history) + 5, 60)
W, B = np.meshgrid(w_grid, b_grid)
Cost = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        y_pred = W[i, j] * x + B[i, j]
        Cost[i, j] = (1 / m) * np.sum((y_pred - y) ** 2)

# Panel 6: line fit
x_line = np.linspace(x.min() - 1, x.max() + 1, 100)

# 3. BUILD THE 2x3 FIGURE
fig = plt.figure(figsize=(12, 7))

ax_cost = fig.add_subplot(2, 3, 1)                    
ax_w = fig.add_subplot(2, 3, 2)                        
ax_b = fig.add_subplot(2, 3, 3)                         
ax_3d = fig.add_subplot(2, 3, 4, projection='3d')       
ax_contour = fig.add_subplot(2, 3, 5)                   
ax_fit = fig.add_subplot(2, 3, 6)                      

# ---- Panel 1: Cost vs Iteration ----
ax_cost.set_xlim(0, len(cost_history))
ax_cost.set_ylim(0, max(cost_history) * 1.05)
ax_cost.set_xlabel("Iteration")
ax_cost.set_ylabel("Cost (MSE)")
ax_cost.set_title("Cost vs Iteration")
cost_line, = ax_cost.plot([], [], color='red', lw=2)

# ---- Panel 2: Cost vs w ----
ax_w.plot(w_range, cost_for_w, color='blue', marker='o', markersize=2, label='Cost landscape')
w_path_line, = ax_w.plot([], [], color='red', lw=1, label='GD path')
w_point, = ax_w.plot([], [], 'go', markersize=8, label='Current')
ax_w.set_xlabel("w")
ax_w.set_ylabel("Cost (MSE)")
ax_w.set_title("Cost vs Weight (w)")
ax_w.legend()

# ---- Panel 3: Cost vs b ----
ax_b.plot(b_range, cost_for_b, color='blue', marker='o', markersize=2, label='Cost landscape')
b_path_line, = ax_b.plot([], [], color='red', lw=1, label='GD path')
b_point, = ax_b.plot([], [], 'go', markersize=8, label='Current')
ax_b.set_xlabel("b")
ax_b.set_ylabel("Cost (MSE)")
ax_b.set_title("Cost vs Bias (b)")
ax_b.legend()

# ---- Panel 4: 3D cost surface ----
ax_3d.plot_surface(W, B, Cost, cmap='viridis', alpha=0.5)
surf_path, = ax_3d.plot([], [], [], color='red', lw=2, label='GD path')
surf_point, = ax_3d.plot([], [], [], 'go', markersize=6, label='Current')
ax_3d.set_xlabel("w")
ax_3d.set_ylabel("b")
ax_3d.set_zlabel("Cost (MSE)")
ax_3d.set_title("Cost Surface (w & b)")
ax_3d.legend()

# ---- Panel 5: Contour (top-down) ----
contour = ax_contour.contour(W, B, Cost, levels=30, cmap='viridis')
ax_contour.clabel(contour, inline=True, fontsize=7)
contour_path, = ax_contour.plot([], [], color='red', lw=1, label='GD path')
contour_point, = ax_contour.plot([], [], 'go', markersize=8, label='Current')
ax_contour.plot(w_history[0], b_history[0], 'bs', markersize=10, label='Start')
ax_contour.set_xlabel("w")
ax_contour.set_ylabel("b")
ax_contour.set_title("Cost Contour")
ax_contour.legend()

# ---- Panel 6: Line fit ----
ax_fit.scatter(x, y, color='blue', label='Actual data', zorder=3)
fit_line, = ax_fit.plot([], [], color='red', lw=2, label='Fitted line')
ax_fit.set_xlim(x.min() - 1, x.max() + 1)
ax_fit.set_ylim(y.min() - 10, y.max() + 10)
ax_fit.set_xlabel("x")
ax_fit.set_ylabel("y")
ax_fit.set_title("Line Fit")
ax_fit.legend()

plt.tight_layout()

# 4. ONE UPDATE FUNCTION DRIVING ALL 6 PANELS TOGETHER
def update(i):
    # Panel 1: cost vs iteration
    cost_line.set_data(range(i + 1), cost_history[:i + 1])

    # Panel 2: cost vs w
    w_path_line.set_data(w_history[:i + 1], cost_history[:i + 1])
    w_point.set_data([w_history[i]], [cost_history[i]])

    # Panel 3: cost vs b
    b_path_line.set_data(b_history[:i + 1], cost_history[:i + 1])
    b_point.set_data([b_history[i]], [cost_history[i]])

    # Panel 4: 3D surface path
    surf_path.set_data(w_history[:i + 1], b_history[:i + 1])
    surf_path.set_3d_properties(cost_history[:i + 1])
    surf_point.set_data([w_history[i]], [b_history[i]])
    surf_point.set_3d_properties([cost_history[i]])

    # Panel 5: contour path
    contour_path.set_data(w_history[:i + 1], b_history[:i + 1])
    contour_point.set_data([w_history[i]], [b_history[i]])

    # Panel 6: line fit
    current_w = w_history[i]
    current_b = b_history[i]
    fit_line.set_data(x_line, current_w * x_line + current_b)
    ax_fit.set_title(f"Line Fit | w={current_w:.2f}, b={current_b:.2f}")

    fig.suptitle(f"Gradient Descent — Iteration {i}", fontsize=14)

    return (cost_line, w_path_line, w_point, b_path_line, b_point,
            surf_path, surf_point, contour_path, contour_point, fit_line)


ani = FuncAnimation(fig, update, frames=frames, interval=50, blit=False)


# 5. SAVE AS GIF (STEP 4 FIX) — instead of embedding inline HTML
# Requires: pip install pillow
ani.save("gradient_descent.gif", writer="pillow", fps=15)
print("Saved animation to gradient_descent.gif")

plt.close(fig)  # prevents a duplicate static figure from also appearing

# STEP 5 FIX: the heavy inline embed is disabled. Uncomment ONLY if you
# specifically need to preview it inside the notebook (this is what
# caused the 56 MB file in the first place):
# from IPython.display import HTML, display
# display(HTML(ani.to_jshtml()))

Final weight: 18.3296, Final bias: -2.7457
Saved animation to gradient_descent.gif


In [5]:
n_inputs = 2      
n_hidden = 2       

np.random.seed(42)
W1 = np.random.randn(n_inputs, n_hidden)   
print(W1)

[[ 0.49671415 -0.1382643 ]
 [ 0.64768854  1.52302986]]
